# Step 3D - Same Subject Locality Stress

This notebook stays on `dev_tune_200`.

It stress-tests the **subject gate** on prompts that contain the edited subject but are not the rewrite/paraphrase task. This matters because the standard near/far locality buckets rarely activate the subject gate.

It does not touch:

```text
analysis_500
ablation_500
final_test_500
final_test_full
```

The derived stress input is written outside the frozen `protocol/` directory.

In [ ]:
%%bash
set -euo pipefail

pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"

In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import subprocess
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "git_commit": commit,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step3d.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)
print("Wrote:", out)
print(json.dumps(env, indent=2))

## Preflight

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "llada_counterfact_protocol.py",
    "llada_experiment_reports.py",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json",
    "runs/counterfact_direction1_v1/dev_tune_200_matched_kl_sweep_subject_v1/report_summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_subject_v1/report_summary.json",
]
for name in required_files:
    path = PROJECT_DIR / name
    assert path.exists(), f"Missing required file: {path}"

thresholds = json.load((PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json").open())
assert 0.0 < float(thresholds["selfloc_base"]) < 1.0
print("Preflight OK.")

## Overwrite Guards

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/same_subject_stress_inputs",
    project / "runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_base",
    project / "runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_prompt_memory_subject",
    project / "runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
    project / "runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_subject_gs200",
    project / "runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_report_v1",
]
for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )
print("Overwrite guard passed.")

## Build Same-Subject Stress Input

In [ ]:
import copy
import json
import random
import sys
from pathlib import Path
from collections import Counter

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from llada_counterfact_protocol import normalize_counterfact_text, render_counterfact_prompt

src = project / "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl"
rows = [json.loads(line) for line in src.open("r", encoding="utf-8") if line.strip()]
assert len(rows) == 200

rng = random.Random(0)
templates = [
    {
        "relation_id": row.get("relation_id"),
        "rewrite_template": row.get("rewrite_template"),
        "subject": row.get("subject"),
    }
    for row in rows
    if row.get("rewrite_template") and row.get("subject")
]

stress_rows = []
stress_counts = Counter()
for row in rows:
    subject = str(row["subject"])
    relation_id = str(row.get("relation_id", ""))
    assert str(row.get("target", "")).strip(), f"Missing target for {row['id']}"
    requested = row.get("requested_rewrite") or {}
    expected_target_new = None
    if isinstance(requested, dict):
        expected_target_new = ((requested.get("target_new") or {}).get("str"))
    if expected_target_new is None and isinstance(row.get("target_new"), dict):
        expected_target_new = row["target_new"].get("text") or row["target_new"].get("str")
    if expected_target_new is not None:
        assert normalize_counterfact_text(row["target"]) == normalize_counterfact_text(expected_target_new), (
            f"row['target'] is not target_new for {row['id']}: "
            f"{row['target']!r} vs {expected_target_new!r}"
        )
    stress_cases = []

    candidates = [
        item for item in templates
        if item.get("relation_id") != relation_id
        and item.get("rewrite_template") != row.get("rewrite_template")
    ]
    rng.shuffle(candidates)
    for idx, item in enumerate(candidates[:3]):
        try:
            prompt = render_counterfact_prompt(str(item["rewrite_template"]), subject)
        except Exception:
            continue
        if prompt.strip() == str(row["prompt"]).strip():
            continue
        stress_cases.append({
            "id": f"{row['id']}_same_subject_template_{idx}",
            "prompt": prompt,
            "target": row["target"],
            "aliases": row.get("aliases", [row["target"]]),
        })
        stress_counts["same_subject_template"] += 1

    for source_name, limit in [("attribute_prompts", 2), ("generation_prompts", 2)]:
        kept = 0
        for prompt in row.get(source_name, []):
            prompt = str(prompt)
            if subject.lower() not in prompt.lower():
                continue
            stress_type = source_name.replace("_prompts", "")
            stress_cases.append({
                "id": f"{row['id']}_{stress_type}_{kept}",
                "prompt": prompt,
                "target": row["target"],
                "aliases": row.get("aliases", [row["target"]]),
            })
            stress_counts[stress_type] += 1
            kept += 1
            if kept >= limit:
                break

    assert stress_cases, f"No stress cases for {row['id']}"
    out_row = copy.deepcopy(row)
    out_row["id"] = f"{row['id']}_same_subject_stress"
    out_row["case_id"] = row["case_id"]
    out_row["source_edit_id"] = row["id"]
    out_row["split_role"] = "dev_tune_200"
    out_row["declarative_paraphrase_prompts"] = []
    out_row["qa_paraphrase_prompts"] = []
    out_row["near_locality_cases"] = stress_cases
    out_row["far_locality_cases"] = []
    out_row["stress_metadata"] = {
        "source": "dev_tune_200",
        "stress_case_count": len(stress_cases),
        "case_ids": [case["id"] for case in stress_cases],
    }
    stress_rows.append(out_row)

out_dir = project / "runs/counterfact_direction1_v1/same_subject_stress_inputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "dev_tune_200_same_subject_stress.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for row in stress_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "protocol_version": "counterfact_direction1_v1",
    "split_role": "dev_tune_200",
    "source_manifest": str(src),
    "stress_input": str(out_path),
    "num_edits": len(stress_rows),
    "stress_counts": dict(stress_counts),
    "analysis_500_used": False,
    "final_test_used": False,
}
with (out_dir / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Wrote:", out_path)
print(json.dumps(summary, indent=2))

## Base Stress Run

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_base

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_base \
  --methods base \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --guidance_scale 1.0 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_base/stdout.log

## Prompt Memory Subject-Gated Stress Run

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_prompt_memory_subject

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_prompt_memory_subject \
  --methods prompt_memory_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --guidance_scale 1.0 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_prompt_memory_subject/stdout.log

## Myopic And No-Rollout Subject-Gated Stress Run

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_bridge_controls_subject_gs200

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_bridge_controls_subject_gs200 \
  --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --guidance_scale 2.0 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_bridge_controls_subject_gs200/stdout.log

## MC Bridge Subject-Gated Stress Run

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_subject_gs200

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_subject_gs200 \
  --methods mc_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 2 \
  --guidance_scale 2.0 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_subject_gs200/stdout.log

## Validate Stress Runs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
expected_runs = {
    "dev_tune_200_same_subject_stress_base": ["base"],
    "dev_tune_200_same_subject_stress_prompt_memory_subject": ["prompt_memory_gated_subject"],
    "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200": ["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"],
    "dev_tune_200_same_subject_stress_mc_bridge_subject_gs200": ["mc_bridge_gated_subject"],
}

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def mean(values):
    values = [float(v) for v in values if v is not None]
    return sum(values) / len(values) if values else None

for run_name, methods in expected_runs.items():
    run_dir = project / "runs/counterfact_direction1_v1" / run_name
    cfg = json.load((run_dir / "run_config.json").open())
    assert cfg["protocol_version"] == "counterfact_direction1_v1"
    assert cfg["split_role"] == "dev_tune_200"
    assert cfg["methods"] == methods
    assert "analysis_500" not in str(cfg)
    assert "final_test" not in str(cfg)
    rows = list(read_jsonl(run_dir / "per_case_results.jsonl"))
    by_method_bucket = defaultdict(set)
    gate_values = defaultdict(list)
    for row in rows:
        method = row.get("method_variant") or row.get("method")
        if method in methods:
            by_method_bucket[(method, row["bucket"])].add(str(row.get("edit_id") or row.get("case_id")))
            if row.get("bucket") == "near_locality" and row.get("gate_activation_rate") is not None:
                gate_values[method].append(float(row["gate_activation_rate"]))
    for method in methods:
        assert len(by_method_bucket[(method, "rewrite")]) == 200
        assert len(by_method_bucket[(method, "near_locality")]) == 200
        if method != "base":
            activation = mean(gate_values[method])
            assert activation is not None, f"Missing gate activation for {method}"
            assert activation >= 0.95, f"{method} did not activate enough on stress prompts: {activation}"
            print(method, "stress gate activation:", activation)
print("Stress run configs, completeness checks, and gate activation checks OK.")

## Build Same-Subject Stress Report

In [ ]:
import csv
import json
import sys
from pathlib import Path
from collections import defaultdict

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_report_v1"
out_dir.mkdir(parents=True, exist_ok=True)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from llada_experiment_reports import paired_bootstrap_delta_by_case

thresholds = json.load((project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json").open())
standard_near_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["near_locality"])
standard_malformed_budget = float(thresholds["malformed_span_rate_budget"])

run_specs = [
    ("base", "base", "dev_tune_200_same_subject_stress_base", "base"),
    ("prompt_memory_gated_subject", "prompt_memory", "dev_tune_200_same_subject_stress_prompt_memory_subject", "prompt_memory_gated_subject"),
    ("myopic_score_gated_subject_gs2.0", "myopic_score", "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200", "myopic_score_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs2.0", "no_rollout_bridge", "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200", "no_rollout_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs2.0", "mc_bridge", "dev_tune_200_same_subject_stress_mc_bridge_subject_gs200", "mc_bridge_gated_subject"),
]

comparison_pairs = [
    ("prompt_memory_gated_subject", "base"),
    ("myopic_score_gated_subject_gs2.0", "base"),
    ("no_rollout_bridge_gated_subject_gs2.0", "base"),
    ("mc_bridge_gated_subject_gs2.0", "base"),
    ("mc_bridge_gated_subject_gs2.0", "myopic_score_gated_subject_gs2.0"),
    ("mc_bridge_gated_subject_gs2.0", "no_rollout_bridge_gated_subject_gs2.0"),
    ("mc_bridge_gated_subject_gs2.0", "prompt_memory_gated_subject"),
]

bootstrap_metrics = [
    "target_false_positive_rate",
    "exact_rate",
    "malformed_rate",
    "token_f1",
    "sparse_guidance_kl",
]

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def mean_or_none(values):
    vals = [float(v) for v in values if v is not None]
    return sum(vals) / len(vals) if vals else None

def stress_type(prompt_id):
    text = str(prompt_id)
    if "_same_subject_template_" in text:
        return "same_subject_template"
    if "_attribute_" in text:
        return "attribute"
    if "_generation_" in text:
        return "generation"
    return "unknown"

def write_csv(path, rows):
    if not rows:
        path.write_text("")
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

all_method_rows = {}
summary_rows = []
sample_rows = []
base_by_type = {}

for label, family, run_name, method in run_specs:
    rows = [
        {**row, "method_variant": label}
        for row in read_jsonl(project / "runs/counterfact_direction1_v1" / run_name / "per_case_results.jsonl")
        if (row.get("method_variant") or row.get("method")) == method
    ]
    all_method_rows[label] = rows
    stress_rows = [row for row in rows if row.get("bucket") == "near_locality"]
    grouped = defaultdict(list)
    for row in stress_rows:
        grouped[(stress_type(row.get("prompt_id")), str(row.get("edit_id") or row.get("case_id")))].append(row)
    by_type = defaultdict(list)
    for (kind, edit_id), items in grouped.items():
        by_type[kind].append({
            "target_new_exact_rate": mean_or_none(item.get("exact_rate") for item in items),
            "target_false_positive_rate": mean_or_none(item.get("target_false_positive_rate") for item in items),
            "malformed_rate": mean_or_none(item.get("malformed_rate") for item in items),
            "sparse_guidance_kl": mean_or_none(item.get("sparse_guidance_kl") for item in items),
            "gate_activation_rate": mean_or_none(item.get("gate_activation_rate") for item in items),
            "token_f1": mean_or_none(item.get("token_f1") for item in items),
            "num_prompt_rows": len(items),
        })
    for kind, values in sorted(by_type.items()):
        base_stats = base_by_type.get(kind, {})
        if label == "base":
            base_stats = {
                "base_target_false_positive_rate": mean_or_none(v["target_false_positive_rate"] for v in values),
                "base_malformed_rate": mean_or_none(v["malformed_rate"] for v in values),
                "base_target_new_exact_rate": mean_or_none(v["target_new_exact_rate"] for v in values),
            }
            base_by_type[kind] = base_stats
        stress_tfpr_budget = (base_stats.get("base_target_false_positive_rate") or 0.0) + 0.03
        stress_malformed_budget = max(0.05, (base_stats.get("base_malformed_rate") or 0.0) + 0.02)
        mean_tfpr = mean_or_none(v["target_false_positive_rate"] for v in values)
        mean_malformed = mean_or_none(v["malformed_rate"] for v in values)
        row = {
            "label": label,
            "family": family,
            "stress_type": kind,
            "num_edits": len(values),
            "num_prompt_rows": sum(v["num_prompt_rows"] for v in values),
            "mean_target_new_exact_rate": mean_or_none(v["target_new_exact_rate"] for v in values),
            "mean_target_false_positive_rate": mean_tfpr,
            "mean_malformed_rate": mean_malformed,
            "mean_sparse_guidance_kl": mean_or_none(v["sparse_guidance_kl"] for v in values),
            "mean_gate_activation_rate": mean_or_none(v["gate_activation_rate"] for v in values),
            "mean_token_f1_to_target_new": mean_or_none(v["token_f1"] for v in values),
            **base_stats,
            "delta_target_false_positive_rate_vs_base": None if mean_tfpr is None or base_stats.get("base_target_false_positive_rate") is None else mean_tfpr - base_stats["base_target_false_positive_rate"],
            "delta_malformed_rate_vs_base": None if mean_malformed is None or base_stats.get("base_malformed_rate") is None else mean_malformed - base_stats["base_malformed_rate"],
            "standard_near_tfpr_budget": standard_near_tfpr_budget,
            "standard_malformed_budget": standard_malformed_budget,
            "stress_relative_tfpr_budget": stress_tfpr_budget,
            "stress_relative_malformed_budget": stress_malformed_budget,
        }
        row["standard_near_budget_pass"] = (
            (row["mean_target_false_positive_rate"] is None or row["mean_target_false_positive_rate"] <= standard_near_tfpr_budget)
            and (row["mean_malformed_rate"] is None or row["mean_malformed_rate"] <= standard_malformed_budget)
        )
        row["stress_relative_budget_pass"] = (
            (row["mean_target_false_positive_rate"] is None or row["mean_target_false_positive_rate"] <= stress_tfpr_budget)
            and (row["mean_malformed_rate"] is None or row["mean_malformed_rate"] <= stress_malformed_budget)
        )
        summary_rows.append(row)
    for row in stress_rows[:20]:
        sample_rows.append({
            "label": label,
            "stress_type": stress_type(row.get("prompt_id")),
            "edit_id": row.get("edit_id"),
            "prompt_id": row.get("prompt_id"),
            "prompt": row.get("prompt"),
            "sample_outputs": json.dumps(row.get("sample_outputs", []), ensure_ascii=False),
            "target": row.get("target"),
            "target_fp": row.get("target_false_positive_rate"),
            "malformed": row.get("malformed_rate"),
            "gate_activation": row.get("gate_activation_rate"),
        })

bootstrap_rows = []
for candidate, baseline in comparison_pairs:
    pair_rows = all_method_rows[candidate] + all_method_rows[baseline]
    for metric in bootstrap_metrics:
        boot = paired_bootstrap_delta_by_case(
            pair_rows,
            candidate_method=candidate,
            baseline_method=baseline,
            bucket="near_locality",
            metric=metric,
            samples=1000,
            seed=0,
        )
        if boot:
            bootstrap_rows.append({
                "candidate": candidate,
                "baseline": baseline,
                "bucket": "near_locality",
                "metric": metric,
                "mean_delta_candidate_minus_baseline": boot["mean_delta"],
                "ci_lower": boot["ci_low"],
                "ci_upper": boot["ci_high"],
                "num_edits": boot["num_edits"],
            })

selected_dense = None
best_by_kl = project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_subject_v1/best_by_kl_budget.csv"
if best_by_kl.exists():
    selected_dense = str(best_by_kl)

write_csv(out_dir / "same_subject_stress_summary.csv", summary_rows)
write_csv(out_dir / "same_subject_stress_output_samples.csv", sample_rows)
write_csv(out_dir / "same_subject_stress_paired_bootstrap.csv", bootstrap_rows)
report = {
    "protocol_version": "counterfact_direction1_v1",
    "split_role": "dev_tune_200",
    "gate_mode": "subject",
    "stress_definition": "Derived dev-only locality cases whose target is set to target_new so exact/target_false_positive measure target over-injection.",
    "standard_near_tfpr_budget": standard_near_tfpr_budget,
    "standard_malformed_budget": standard_malformed_budget,
    "stress_budget_rule": "stress_tfpr_budget = base_stress_tfpr + 0.03; stress_malformed_budget = max(0.05, base_stress_malformed + 0.02)",
    "selected_dense_pareto_reference": selected_dense,
    "analysis_500_used": False,
    "final_test_used": False,
    "artifacts": {
        "summary": str(out_dir / "same_subject_stress_summary.csv"),
        "samples": str(out_dir / "same_subject_stress_output_samples.csv"),
        "paired_bootstrap": str(out_dir / "same_subject_stress_paired_bootstrap.csv"),
    },
}
with (out_dir / "report_summary.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("Wrote:", out_dir)
for row in summary_rows:
    print(row)


## Expected Outcome

Inspect first:

```text
same_subject_stress_summary.csv
same_subject_stress_paired_bootstrap.csv
same_subject_stress_output_samples.csv
report_summary.json
```

This stress test is intentionally stricter than the standard near/far locality buckets. A high target-new exact rate here means the subject gate is over-activating on same-subject non-edit prompts.

Use both budget views:

```text
standard_near_budget_pass
stress_relative_budget_pass
```

The stress-relative view is the important one for this derived distribution because it measures extra target leakage over base on the same stress prompts.